## Import Libraries and CSV Data

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
#from sklearn.model_selection import GridSearchCV

df = pd.read_csv("csv's/mlb_2024-2026.csv")

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{tuple(df.columns)}")
print(f"\nDate range: {df['game_date'].min()} → {df['game_date'].max()}")
print(f"\nPitch type distribution:\n{df['pitch_type'].value_counts()}")

Shape: (1824142, 15)

Columns:
('game_date', 'player_name', 'pitcher', 'p_throws', 'pitch_type', 'release_speed', 'release_spin_rate', 'pfx_z', 'pfx_x', 'release_extension', 'release_pos_z', 'release_pos_x', 'effective_speed', 'spin_axis', 'delta_run_exp')

Date range: 2024-03-01 → 2026-05-30

Pitch type distribution:
pitch_type
FF    554903
SI    276069
SL    254799
CH    184339
FC    139235
ST    130246
CU    113681
FS     55949
KC     30346
SV      8958
EP      1633
FA      1449
KN      1299
FO      1243
CS       495
SC       214
PO       128
UN        19
Name: count, dtype: int64


# Clean Data

In [2]:
STAT_MAP = {
    'release_spin_rate' : 'spin_rate',
    'pfx_z'             : 'induced_vertical_break',
    'pfx_x'             : 'horizontal_break',
    'release_extension' : 'extension',
    'release_pos_z'     : 'release_height',
    'release_pos_x'     : 'release_side',
    'release_spin_axis' : 'spin_axis',
}

FEATURES = [
    'release_speed',
    'spin_rate',
    'induced_vertical_break',
    'horizontal_break',
    'extension',
    'release_height',
    'release_side',
    'spin_axis',
]

FLIP_COLS = ['horizontal_break', 'release_side']

# Unnecessary pitch types
DROP_TYPES = {'SV', 'EP', 'FA', 'CS', 'SC', 'PO', 'UN'}

# Map Statcast pitch types to Trackman pitch types
PITCH_MAP = {
    'FF': 'FB',
    'CU': 'CB',
    'KC': 'CB',  # knuckle-curve treated as CB
    'FC': 'CT',
    'ST': 'SW',
    'FS': 'SP',
    'FO': 'SP',  # forkball treated as SP
}

df = df.rename(columns=STAT_MAP)

# Normalize LHP → RHP convention
df.loc[df['p_throws'] == 'L', FLIP_COLS] *= -1
df.loc[df['p_throws'] == 'L', 'spin_axis'] = (360 - df.loc[df['p_throws'] == 'L', 'spin_axis']) % 360

# Drop rows missing the target variable
df = df.dropna(subset=['delta_run_exp']).copy()

# Drop unnecessary pitch types
df = df[~df['pitch_type'].isin(DROP_TYPES)]

# Merge certain pitches
df['pitch_type'] = df['pitch_type'].replace(PITCH_MAP)

print(f"Rows after cleaning: {len(df):,}")
print(f"\nNull counts:\n{df[FEATURES + ['delta_run_exp']].isnull().sum()}")
print(f"\nPitch type counts after cleaning:\n{df['pitch_type'].value_counts()}")

Rows after cleaning: 1,805,872

Null counts:
release_speed             63507
spin_rate                 70675
induced_vertical_break    63506
horizontal_break          63639
extension                 65577
release_height            63506
release_side              63506
spin_axis                 70679
delta_run_exp                 0
dtype: int64

Pitch type counts after cleaning:
pitch_type
FB    554896
SI    276064
SL    254791
CH    184334
CB    144024
CT    139231
SW    130242
SP     57190
KN      1299
Name: count, dtype: int64


## Compute Differentials

In [ ]:
FB_TYPES = {'FB', 'SI'}

fb_avg = (
    df[df['pitch_type'].isin(FB_TYPES)]
    .groupby('pitcher')[['release_speed', 'induced_vertical_break',
                               'horizontal_break', 'spin_rate']]
    .mean()
    .rename(columns={
        'release_speed':          'fb_velo',
        'induced_vertical_break': 'fb_ivb',
        'horizontal_break':       'fb_hb',
        'spin_rate':              'fb_spin',
    })
)

df = df.join(fb_avg, on='pitcher_name')

df['velo_diff'] = df['fb_velo'] - df['release_speed']
df['ivb_diff']  = df['fb_ivb']  - df['induced_vertical_break']
df['hb_diff']   = df['fb_hb']   - df['horizontal_break']
df['spin_diff'] = df['fb_spin'] - df['spin_rate']

fb_mask = df['pitch_type'].isin(FB_TYPES)
df.loc[fb_mask, ['velo_diff', 'ivb_diff', 'hb_diff', 'spin_diff']] = 0

df.drop(columns=['fb_velo', 'fb_ivb', 'fb_hb', 'fb_spin'], inplace=True)

# ── Updated feature list ───────────────────────────────────────────────────
FEATURES = [
    'release_speed',
    'spin_rate',
    'induced_vertical_break',
    'horizontal_break',
    'extension',
    'release_height',
    'release_side',
    'spin_axis',
    'velo_diff',
    'ivb_diff',
    'hb_diff',
    'spin_diff',
]

KeyError: 'pitcher_name'

# Train Model

In [ ]:
for pitch_type, group in df.groupby('pitch_type'):
    # Define inputs and outputs
    X = group[FEATURES]
    y = group['delta_run_exp']

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 24)

    # Set hyperparameters (Use gridsearch later to tune these better)
    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=24
        )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    # Uncomment this line to save the models again if necessary
    model.save_model(f'models/v2/{pitch_type}.ubj')

    print(f"{pitch_type}: n = {len(group):,} RMSE = {rmse:.4f}")

print("\nModels saved to /models directory")

CB: n = 143,232 RMSE = 0.2111
CH: n = 181,021 RMSE = 0.2233
CT: n = 137,654 RMSE = 0.2398
FB: n = 553,840 RMSE = 0.2278
KN: n = 1,378 RMSE = 0.2090
SI: n = 276,048 RMSE = 0.2257
SL: n = 254,143 RMSE = 0.2247
SP: n = 57,673 RMSE = 0.2187
SW: n = 130,248 RMSE = 0.2212

Models saved to /models directory
